# Soluciones — Clase 15: SQL básico con Python

Este cuaderno contiene las soluciones a los ejercicios de la Clase 15.

In [ ]:
# Configuración inicial — ejecutar primero
# Qué hace: prepara la base de datos en memoria con los datos de ventas
# Para qué sirve: tener el entorno listo para todas las soluciones

import sqlite3
import pandas as pd

df = pd.read_csv("../../datasets/ventas_tienda.csv")
conn = sqlite3.connect(":memory:")
df.to_sql("ventas", conn, if_exists="replace", index=False)
print("Base de datos lista. Registros:", len(df))

In [ ]:
# Solución — Ejercicio 1: SELECT básico
# Qué hace: demuestra las cuatro variantes de SELECT pedidas
# Para qué sirve: practicar la lectura básica de datos con SQL

# a) Primeras 10 filas, todas las columnas
print("=== a) Primeras 10 filas ===")
print(pd.read_sql("SELECT * FROM ventas LIMIT 10", conn))

# b) Columnas específicas, 15 filas
print("\n=== b) Columnas seleccionadas ===")
print(pd.read_sql("""
    SELECT producto, precio_unitario, unidades
    FROM ventas
    LIMIT 15
""", conn))

# c) COUNT total de registros
print("\n=== c) Total de registros ===")
print(pd.read_sql("SELECT COUNT(*) AS total FROM ventas", conn))

# d) Valores únicos de categoría
print("\n=== d) Categorías únicas ===")
print(pd.read_sql("SELECT DISTINCT categoria FROM ventas ORDER BY categoria", conn))

In [ ]:
# Solución — Ejercicio 2: WHERE
# Qué hace: aplica distintos tipos de filtros con WHERE
# Para qué sirve: practicar el equivalente SQL de los filtros de pandas

# a) precio_unitario > 40
r = pd.read_sql("SELECT * FROM ventas WHERE precio_unitario > 40", conn)
print(f"a) precio_unitario > 40: {len(r)} registros")

# b) unidades <= 5
r = pd.read_sql("SELECT * FROM ventas WHERE unidades <= 5", conn)
print(f"b) unidades <= 5: {len(r)} registros")

# c) BETWEEN
r = pd.read_sql("SELECT * FROM ventas WHERE precio_unitario BETWEEN 20 AND 60", conn)
print(f"c) precio BETWEEN 20 AND 60: {len(r)} registros")

# d) categoría específica (tomamos la primera que exista)
primera_categoria = df['categoria'].iloc[0]
r = pd.read_sql(f"SELECT * FROM ventas WHERE categoria = '{primera_categoria}'", conn)
print(f"d) categoria = '{primera_categoria}': {len(r)} registros")

# e) AND combinado
r = pd.read_sql("""
    SELECT producto, precio_unitario, unidades
    FROM ventas
    WHERE precio_unitario > 30 AND unidades > 8
""", conn)
print(f"e) precio > 30 AND unidades > 8: {len(r)} registros")
print(r)

In [ ]:
# Solución — Ejercicio 3: GROUP BY
# Qué hace: responde las 5 preguntas de negocio con GROUP BY
# Para qué sirve: demostrar cómo SQL responde preguntas de resumen en pocas líneas

# a) Registros por categoría
print("=== a) Registros por categoría ===")
print(pd.read_sql("SELECT categoria, COUNT(*) AS registros FROM ventas GROUP BY categoria", conn))

# b) Precio promedio por categoría
print("\n=== b) Precio promedio por categoría ===")
print(pd.read_sql("""
    SELECT categoria, ROUND(AVG(precio_unitario), 2) AS precio_promedio
    FROM ventas
    GROUP BY categoria
    ORDER BY precio_promedio DESC
""", conn))

# c) Unidades totales por categoría
print("\n=== c) Unidades totales por categoría ===")
print(pd.read_sql("""
    SELECT categoria, SUM(unidades) AS total_unidades
    FROM ventas
    GROUP BY categoria
    ORDER BY total_unidades DESC
""", conn))

# d) Precio máximo y mínimo por categoría
print("\n=== d) MAX y MIN por categoría ===")
print(pd.read_sql("""
    SELECT
        categoria,
        MAX(precio_unitario) AS precio_maximo,
        MIN(precio_unitario) AS precio_minimo
    FROM ventas
    GROUP BY categoria
""", conn))

# e) Ordenado por suma de unidades
print("\n=== e) Ordenado por suma de unidades ===")
print(pd.read_sql("""
    SELECT categoria, SUM(unidades) AS total_unidades
    FROM ventas
    GROUP BY categoria
    ORDER BY total_unidades DESC
""", conn))

In [ ]:
# Solución — Ejercicio 4: Comparación pandas vs SQL
# Qué hace: muestra las tres operaciones en ambos lenguajes con resultados equivalentes
# Para qué sirve: consolidar la equivalencia entre pandas y SQL

# a) Filtrar precio_unitario > 50
print("=== a) Filtrar precio_unitario > 50 ===")
sql = pd.read_sql("SELECT * FROM ventas WHERE precio_unitario > 50", conn)
pnd = df[df['precio_unitario'] > 50].reset_index(drop=True)
print(f"SQL: {len(sql)} filas | pandas: {len(pnd)} filas — ¿Iguales? {len(sql) == len(pnd)}")

# b) Promedio de precio_unitario por categoria
print("\n=== b) Promedio por categoría ===")
sql = pd.read_sql("""
    SELECT categoria, AVG(precio_unitario) AS precio_promedio
    FROM ventas GROUP BY categoria ORDER BY categoria
""", conn)
pnd = df.groupby('categoria')['precio_unitario'].mean().reset_index()
pnd.columns = ['categoria', 'precio_promedio']
pnd = pnd.sort_values('categoria').reset_index(drop=True)
print("SQL:")
print(sql)
print("pandas:")
print(pnd)

# c) Top 5 por precio_unitario descendente
print("\n=== c) Top 5 más caros ===")
sql = pd.read_sql("""
    SELECT producto, precio_unitario FROM ventas
    ORDER BY precio_unitario DESC LIMIT 5
""", conn)
pnd = df[['producto','precio_unitario']].sort_values('precio_unitario', ascending=False).head(5).reset_index(drop=True)
print("SQL:", sql.values.tolist())
print("pandas:", pnd.values.tolist())

In [ ]:
# Solución — Ejercicio 5 (desafío): Consulta combinada con HAVING
# Qué hace: combina GROUP BY, múltiples agregaciones, HAVING y ORDER BY
# Para qué sirve: demostrar el poder de SQL para análisis complejos en una sola consulta

query = """
    SELECT
        categoria,
        COUNT(*)                                  AS registros,
        SUM(unidades)                             AS total_unidades,
        ROUND(AVG(precio_unitario), 2)            AS precio_promedio,
        ROUND(SUM(precio_unitario * unidades), 2) AS ingreso_total
    FROM ventas
    GROUP BY categoria
    HAVING COUNT(*) > 5
    ORDER BY ingreso_total DESC
"""

resultado = pd.read_sql(query, conn)
print("Categorías con más de 5 registros, ordenadas por ingreso total:")
print(resultado)

print("\nNota clave:")
print("  WHERE filtra filas ANTES de GROUP BY")
print("  HAVING filtra grupos DESPUÉS de GROUP BY")
print("  En esta consulta, HAVING COUNT(*) > 5 elimina categorías con pocos registros")

In [ ]:
# Cierre — liberar recursos
conn.close()
print("Conexión cerrada.")

## Reflexión de cierre

Con esta clase cerramos el ciclo de herramientas fundamentales del bootcamp:

- **Python**: el lenguaje base, la lógica y el control de flujo
- **pandas**: manipulación de tablas, limpieza, agrupación
- **NumPy**: cálculo numérico vectorizado sobre arrays
- **matplotlib/seaborn**: visualización de patrones
- **scikit-learn**: modelos de machine learning
- **SQL**: el lenguaje para consultar bases de datos reales

Cada una de estas herramientas tiene su lugar en el ciclo CRISP-DM. Ahora tienes el mapa completo.